# News Search Engine — Demo

An end-to-end walkthrough of the information-retrieval pipeline: load the
corpus, build the inverted index, search with BM25, expand queries, and
evaluate. See [`src/news_search`](../src/news_search) for the implementation.

In [ ]:
import sys
sys.path.insert(0, '../src')

from news_search import load_corpus, build_index, SearchEngine
from news_search import evaluate as ev

## 1. Load the corpus

Uses the committed sample by default. Swap the path for
`../data/News_Category_Dataset_v3.json` to run on all ~210k articles.

In [ ]:
docs = load_corpus('../data/sample_news.jsonl')
print(f'Loaded {len(docs):,} documents')
docs[0]

## 2. Build the inverted index

Document lengths, IDF and average document length are precomputed once here,
so queries never re-tokenise documents.

In [ ]:
index = build_index(docs)
print(f'{index.num_docs:,} docs | {index.vocabulary_size:,} terms | avg len {index.avg_doc_len:.1f}')
engine = SearchEngine(index)

## 3. Search (BM25)

BM25 is the default ranker. Try `method='tfidf'` to see the difference —
plain TF-IDF tends to surface very short headlines.

In [ ]:
res = engine.search('covid vaccine health', method='bm25', top_k=5)
print(f'{res.total_hits} hits in {res.elapsed_ms:.2f} ms')
for r in res.results:
    print(f"  #{r['rank']} [{r['category']}] {r['headline'][:60]}  (score {r['score']})")

## 4. Query expansion

Compare the expansion terms each method pulls in. (`bert` / `prf+bert`
require the optional dependencies in `requirements-bert.txt`.)

In [ ]:
for method in ['bm25', 'prf', 'wordnet']:
    r = engine.search('election president', method=method, top_k=5)
    print(f"{method:8} | expansion: {r.expansion_terms}")

## 5. Evaluation

Precision@K / Recall / F1 against category-based pseudo-relevance judgments.
Precision@K is the meaningful metric (see the README for why). BM25 should
clearly beat plain TF-IDF.

In [ ]:
queries = ['covid health', 'election president', 'movie film',
           'stock market', 'travel food', 'game sport']
for s in ev.evaluate(engine, queries, methods=['bm25', 'tfidf', 'prf', 'wordnet'], top_k=10):
    print(f'{s.method:8} P@10={s.precision:.3f}  R={s.recall:.4f}  F1={s.f1:.4f}  {s.avg_ms:.2f}ms')